# Multi-Tissue Comparison Analysis

Compare multiple tissue samples:
- Integrate multiple Xenium datasets
- Batch correction
- Cross-tissue cell type comparison
- Tissue-specific spatial patterns

**Input:** Multiple preprocessed and annotated AnnData objects

**Output:** Integrated multi-tissue dataset with comparative analysis

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sc.settings.set_figure_params(dpi=80)
print(f"Scanpy version: {sc.__version__}")
print(f"scVI version: {scvi.__version__}")

## 1. Load Multiple Tissue Samples

In [ ]:
import os
import gc
from pathlib import Path

SAMPLE_IDS = os.environ.get("XENIUM_SAMPLE_IDS", "0041323,0041326").split(",")
SAMPLE_IDS = [s.strip() for s in SAMPLE_IDS if s.strip()]

PROCESSED_ROOT = Path(os.environ.get("XENIUM_PROCESSED_ROOT", "../data/processed"))
OUTPUT_DIR     = PROCESSED_ROOT / "integrated"
FIGURES_DIR    = Path(os.environ.get("XENIUM_FIGURES_DIR",
                                     f"../figures/integrated_{'_'.join(SAMPLE_IDS)}"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Samples to integrate: {SAMPLE_IDS}")

# Prefer pre-slimmed h5ads (run scripts/slim_phenotyped.py first); fall back to full.
adatas = []
for sample in SAMPLE_IDS:
    slim = PROCESSED_ROOT / sample / f"{sample}_phenotyped_slim.h5ad"
    full = PROCESSED_ROOT / sample / f"{sample}_phenotyped.h5ad"
    src = slim if slim.exists() else full
    if not src.exists():
        raise FileNotFoundError(
            f"Neither {slim} nor {full} exists. Run 02 for sample {sample}, "
            f"then scripts/slim_phenotyped.py."
        )
    ad = sc.read_h5ad(src)
    ad.obs['sample_id'] = sample
    adatas.append(ad)
    print(f"Loaded {sample}: {ad.shape} from {src.name}")

print(f"\nTotal samples loaded: {len(adatas)}")


## 2. Concatenate Samples

In [ ]:
import anndata as ad

if len(adatas) > 1:
    adata_combined = ad.concat(
        adatas,
        axis=0,
        join='inner',
        label='sample_batch',
        keys=SAMPLE_IDS,
        index_unique='-',
        merge='first',   # uns/var merge strategy
    )
    print(f"\nCombined dataset shape: {adata_combined.shape}")
    # Immediately free per-sample adatas
    for i in range(len(adatas)):
        adatas[i] = None
    adatas = []
    gc.collect()
else:
    adata_combined = adatas[0].copy()
    adata_combined.obs['sample_batch'] = SAMPLE_IDS[0]
    print(f"\nSingle sample mode: {adata_combined.shape}")

print(f"Samples: {adata_combined.obs['sample_id'].nunique()}")
print(f"Total cells: {adata_combined.n_obs}")


## 3. Pre-Integration QC

In [ ]:
# Plot cell type composition per sample
composition = pd.crosstab(
    adata_combined.obs['sample_id'],
    adata_combined.obs['celltype'],
    normalize='index'
) * 100

print("\nCell type composition (%) by sample:")
print(composition)

# Plot
composition.T.plot(kind='bar', figsize=(12, 6), stacked=False)
plt.ylabel('Percentage')
plt.xlabel('Cell Type')
plt.title('Cell Type Composition Across Samples')
plt.legend(title='Sample', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'composition_by_sample.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Batch Correction with scVI

In [ ]:
# scVI batch-correction on GPU (full 2.5M cells), then UMAP on a 500k subsample
# (plain scanpy CPU UMAP on 2.5M cells takes many hours; subsample is standard for
# visualization at this scale. scVI latent is still fit on all cells.)
import torch

use_gpu = torch.cuda.is_available()
print(f"CUDA available: {use_gpu}")

if len(SAMPLE_IDS) > 1:
    layer_arg = 'counts' if 'counts' in adata_combined.layers else None
    scvi.model.SCVI.setup_anndata(
        adata_combined,
        layer=layer_arg,
        batch_key='sample_batch',
    )
    vae = scvi.model.SCVI(
        adata_combined,
        n_layers=2,
        n_latent=30,
        gene_likelihood="nb",
    )
    print("Training scVI for batch correction (full 2.5M cells) ...")
    vae.train(
        max_epochs=120,
        early_stopping=True,
        early_stopping_patience=25,
        accelerator=("gpu" if use_gpu else "cpu"),
        devices=1,
        batch_size=4096,
        datasplitter_kwargs={"num_workers": 16, "pin_memory": True, "persistent_workers": True},
    )
    adata_combined.obsm['X_scvi'] = vae.get_latent_representation()
    print("scVI done. X_scvi available on all cells.")

    # Stratified subsample for UMAP visualization (per sample_batch)
    N_UMAP = 500_000
    if adata_combined.n_obs > N_UMAP:
        rng = np.random.default_rng(0)
        counts_per = adata_combined.obs['sample_batch'].value_counts()
        fracs = counts_per / counts_per.sum()
        n_per = (fracs * N_UMAP).round().astype(int)
        n_per.iloc[0] += N_UMAP - n_per.sum()
        keep = []
        for sid, n in n_per.items():
            idx = np.where(adata_combined.obs['sample_batch'] == sid)[0]
            keep.append(rng.choice(idx, size=min(int(n), len(idx)), replace=False))
        sub_idx = np.sort(np.concatenate(keep))
        adata_sub = adata_combined[sub_idx].copy()
        print(f"UMAP subsample: {adata_sub.n_obs:,}/{adata_combined.n_obs:,} cells")
    else:
        adata_sub = adata_combined

    print("CPU neighbors/UMAP/leiden on subsample ...")
    sc.pp.neighbors(adata_sub, use_rep='X_scvi', n_neighbors=50)
    sc.tl.umap(adata_sub, min_dist=0.5, spread=2.0)
    sc.tl.leiden(adata_sub, resolution=0.5, key_added='leiden_integrated')

    # Write UMAP + leiden back to the full adata: the subsampled cells get their values,
    # others get NaN / placeholder.
    full_umap = np.full((adata_combined.n_obs, 2), np.nan, dtype=np.float32)
    full_umap[sub_idx] = adata_sub.obsm['X_umap']
    adata_combined.obsm['X_umap'] = full_umap
    full_leiden = pd.Series(pd.Categorical([pd.NA]*adata_combined.n_obs,
                                           categories=adata_sub.obs['leiden_integrated'].cat.categories),
                            index=adata_combined.obs_names, name='leiden_integrated')
    full_leiden.iloc[sub_idx] = adata_sub.obs['leiden_integrated'].values
    adata_combined.obs['leiden_integrated'] = full_leiden
    # Also keep subsample separately for later figures
    adata_combined.uns['umap_subsample_idx'] = sub_idx
    del adata_sub
    gc.collect()
    print("Integration done.")
else:
    print("Single sample — batch correction not needed")
    if 'X_umap' not in adata_combined.obsm:
        sc.pp.neighbors(adata_combined, n_neighbors=50)
        sc.tl.umap(adata_combined, min_dist=0.5, spread=2.0)


## 5. Visualize Integration

In [ ]:
# Plot UMAP colored by sample and cell type
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

sc.pl.umap(adata_combined, color='sample_id', ax=axes[0, 0], show=False, title='Sample ID')
sc.pl.umap(adata_combined, color='celltype', ax=axes[0, 1], show=False, title='Cell Type')
sc.pl.umap(adata_combined, color='total_counts', ax=axes[1, 0], show=False, cmap='viridis', title='Total Counts')
sc.pl.umap(adata_combined, color='n_genes_by_counts', ax=axes[1, 1], show=False, cmap='viridis', title='N Genes')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'integrated_umap.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Cross-Tissue DE Analysis

In [ ]:
# Find genes differentially expressed across samples
if adata_combined.obs['sample_id'].nunique() > 1:
    # Slim script puts counts in X and drops scaled/log_normalized layers.
    if 'log_normalized' not in adata_combined.layers:
        # Build from X (which holds counts after slim)
        counts = adata_combined.layers.get('counts', adata_combined.X)
        # Create a small adata view for normalize to avoid mutating X on huge matrix
        _tmp = adata_combined.copy()
        if 'counts' in _tmp.layers:
            _tmp.X = _tmp.layers['counts'].copy()
        sc.pp.normalize_total(_tmp)
        sc.pp.log1p(_tmp)
        adata_combined.layers['log_normalized'] = _tmp.X
        del _tmp
        gc.collect()

    # Subsample for DE (2000 cells per sample)
    n_per_group = 2000
    np.random.seed(42)
    indices = []
    for group in adata_combined.obs['sample_id'].unique():
        group_idx = adata_combined.obs.index[adata_combined.obs['sample_id'] == group]
        indices.extend(np.random.choice(group_idx, size=min(n_per_group, len(group_idx)), replace=False))
    adata_de = adata_combined[indices].copy()

    sc.tl.rank_genes_groups(
        adata_de,
        groupby='sample_id',
        method='wilcoxon',
        layer='log_normalized',
        key_added='de_samples',
        use_raw=False,
    )

    # Copy results back to main object
    adata_combined.uns['de_samples'] = adata_de.uns['de_samples']

    # Plot top differential genes
    sc.pl.rank_genes_groups(adata_de, n_genes=20, sharey=False, key='de_samples')
    plt.savefig(FIGURES_DIR / 'cross_sample_de.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('Cross-sample DE analysis completed')
else:
    print("Single sample — skipping cross-sample DE")


## 7. Cell Type-Specific Tissue Differences

In [ ]:
# Analyze tissue differences for each cell type
if adata_combined.obs['sample_id'].nunique() > 1:
    # Ensure log_normalized layer exists
    if 'log_normalized' not in adata_combined.layers:
        adata_combined.X = adata_combined.layers['counts'].copy()
        sc.pp.normalize_total(adata_combined)
        sc.pp.log1p(adata_combined)
        adata_combined.layers['log_normalized'] = adata_combined.X.copy()
        adata_combined.X = adata_combined.layers['scaled'].copy()

    celltype_tissue_de = []

    for ct in adata_combined.obs['celltype'].unique():
        print(f"\nAnalyzing {ct}...")

        # Subset
        adata_ct = adata_combined[adata_combined.obs['celltype'] == ct].copy()

        # Check if we have multiple samples
        if adata_ct.obs['sample_id'].nunique() < 2 or adata_ct.n_obs < 20:
            print(f"  Skipping - insufficient data")
            continue

        # Subsample for DE (1000 cells per group)
        n_per_group = 1000
        np.random.seed(42)
        indices = []
        for group in adata_ct.obs['sample_id'].unique():
            group_idx = adata_ct.obs.index[adata_ct.obs['sample_id'] == group]
            indices.extend(np.random.choice(group_idx, size=min(n_per_group, len(group_idx)), replace=False))
        adata_de = adata_ct[indices].copy()

        # Run DE on subsampled data
        sc.tl.rank_genes_groups(
            adata_de,
            groupby='sample_id',
            method='wilcoxon',
            layer='log_normalized',
            use_raw=False
        )

        # Copy results back
        adata_ct.uns['rank_genes_groups'] = adata_de.uns['rank_genes_groups']

        # Get results
        ct_de = sc.get.rank_genes_groups_df(adata_ct, group=None)
        ct_de['celltype'] = ct
        celltype_tissue_de.append(ct_de)
        print(f"  {len(ct_de)} DE genes found")

    # Combine
    if celltype_tissue_de:
        all_ct_de = pd.concat(celltype_tissue_de, ignore_index=True)
        all_ct_de.to_csv(
            OUTPUT_DIR / 'celltype_tissue_de_genes.csv',
            index=False
        )
        print(f"\nCombined results: {len(all_ct_de)} entries")

## 8. Tissue-Specific Spatial Patterns (if applicable)

In [ ]:
# Compare spatial patterns across tissues
# This requires spatial coordinates to be preserved during concatenation

if 'spatial' in adata_combined.obsm and len(adatas) > 1:
    import squidpy as sq
    
    # Calculate neighborhood enrichment per sample
    for sample in adata_combined.obs['sample_id'].unique():
        print(f"\nAnalyzing spatial patterns in {sample}...")
        
        adata_sample = adata_combined[adata_combined.obs['sample_id'] == sample].copy()
        
        # Build spatial graph
        sq.gr.spatial_neighbors(adata_sample, coord_type='generic', n_neighs=10)
        
        # Neighborhood enrichment
        sq.gr.nhood_enrichment(adata_sample, cluster_key='celltype')
        
        # Plot
        sq.pl.nhood_enrichment(
            adata_sample,
            cluster_key='celltype',
            title=f'{sample} Neighborhood Enrichment'
        )
        plt.savefig(FIGURES_DIR / f'nhood_enrichment_{sample}.png', dpi=300, bbox_inches='tight')
        plt.show()
else:
    print("Spatial coordinates not available for comparison")

## 9. Save Integrated Data

In [ ]:
# Save integrated dataset
output_file = OUTPUT_DIR / f"{'_'.join(SAMPLE_IDS)}_integrated.h5ad"

# Sanitize any uns entries with tuple keys (e.g., ligrec) that can't be h5-serialized
for k in list(adata_combined.uns.keys()):
    v = adata_combined.uns[k]
    if isinstance(v, dict) and any(not isinstance(ki, str) for ki in v.keys()):
        print(f"Dropping uns['{k}'] (non-str dict keys)")
        del adata_combined.uns[k]

try:
    adata_combined.write_h5ad(output_file)
    print(f"\nIntegrated dataset saved to: {output_file}")
except Exception as e:
    print(f"write failed ({e!r}); keeping simple uns only and retrying")
    safe = {k: v for k, v in adata_combined.uns.items()
            if isinstance(v, (str, int, float, bool, list, tuple, dict, np.ndarray))}
    adata_combined.uns = safe
    adata_combined.write_h5ad(output_file)
    print(f"Saved after sanitizing uns: {output_file}")

print(f"Dataset shape: {adata_combined.shape}")
print(f"\nCell type summary:")
print(adata_combined.obs['celltype'].value_counts())
